## tl;dr

The frozen positioning diagnostic is rejected without prospective scoring: its intercept dominates every bounded feature correction, so it raises every probability and cannot satisfy the non-positive correction-on-losses gate. The attempted fresh PMXT source also returned zero target events in the first two processed hours.

## Context & Methods

This diagnostic audits gate feasibility before spending PMXT bandwidth. The model and clip bounds are frozen from the July 15 historical screen. The fresh-source check uses a July 11–13 request, exact five-minute Gamma metadata, one-second Binance tape, 202 ms replay latency, and atomic session-owned parquet cleanup.

### Key Assumptions

- Both standardized features are clipped independently to `[-5, 5]`, as in the frozen scorer.
- The executable market probability lies strictly inside `(0, 1)`.
- The non-positive mean correction on losses is a hard predeclared gate.

## Data

Load the durable feasibility evidence and expose the audited model and fresh-source manifest.

In [1]:
import json
from pathlib import Path

evidence_path = Path('deploy/promotions/evidence/strategy_registry/20260715_frozen_positioning_forward_feasibility_audit.json')
evidence = json.loads(evidence_path.read_text())
model = evidence['frozen_model']
forward = evidence['fresh_forward_attempt']
{
    'model_status': model['status'],
    'requested_hours': forward['requested_hours'],
    'processed_hours': forward['processed_hours_before_fail_closed_stop'],
    'pmxt_target_events': forward['pmxt_target_events'],
    'manifest_complete': forward['manifest_complete'],
}

{'model_status': 'frozen_for_future_diagnostic_only', 'requested_hours': 72, 'processed_hours': 2, 'pmxt_target_events': 0, 'manifest_complete': False}

## Results

Recompute the exact correction interval from the frozen coefficients.

In [2]:
clip_bound = evidence['feasibility_proof']['feature_clip_abs']
coefficients = model['standardized_coefficients']
feature_abs_bound = clip_bound * sum(abs(value) for value in coefficients.values())
minimum_logit_correction = model['intercept'] - feature_abs_bound
maximum_logit_correction = model['intercept'] + feature_abs_bound

recomputed = {
    'intercept_log_odds': model['intercept'],
    'maximum_abs_feature_contribution_log_odds': feature_abs_bound,
    'minimum_possible_logit_correction': minimum_logit_correction,
    'maximum_possible_logit_correction': maximum_logit_correction,
    'loss_gate_feasible': minimum_logit_correction <= 0.0,
}
assert abs(minimum_logit_correction - evidence['feasibility_proof']['minimum_possible_logit_correction']) < 1e-15
assert minimum_logit_correction > 0.0
recomputed

{'intercept_log_odds': 0.07313642333336683, 'maximum_abs_feature_contribution_log_odds': 0.0249354474658351, 'minimum_possible_logit_correction': 0.04820097586753173, 'maximum_possible_logit_correction': 0.09807187079920193, 'loss_gate_feasible': False}

The positive lower bound is decisive: monotonicity of the logistic function makes every probability correction positive, including corrections on losses.

In [3]:
assert evidence['feasibility_proof']['gate_feasible'] is False
assert forward['pmxt_target_events'] == 0
assert forward['manifest_complete'] is False
assert evidence['decision']['positioning_family'] == 'closed_rejected'
print('All fail-closed feasibility and source-availability assertions passed.')

All fail-closed feasibility and source-availability assertions passed.


## Takeaways

- Do not resume the 72-hour PMXT run or refit the positioning model.
- Treat URL availability as insufficient; require non-zero target events and a complete manifest.
- Keep the retained A− strategy unchanged and live trading off.
- The next useful external action is a segmented, converted forward capture after explicit deployment authorization.